# Phase 2: Wildfire Swarm MARL Optimization
[<-- Return to Main README](../README.md) | [<-- Previous: Problem Statement](./01_Problem_Statement_and_Methodology.ipynb)

This notebook contains the live generative environment. Because wildfires are chaotic and unpredictable, we do not use static historical CSV files. Instead, the `WildfireSwarmEnv` mathematically generates a new, randomized fire spread pattern every single time it runs. The AI must learn to adapt to live chaos.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

from wildfire_swarm_env import WildfireSwarmEnv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computational Device (Hardware): {device}")

In [ ]:
class SwarmQNetwork(nn.Module):
    def __init__(self, obs_dim, num_drones):
        super(SwarmQNetwork, self).__init__()
        self.num_actions = 6 ** num_drones
        self.fc = nn.Sequential(
            nn.Linear(obs_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, self.num_actions)
        )
    def forward(self, x): return self.fc(x)

class ReplayBuffer:
    def __init__(self, capacity): self.buffer = deque(maxlen=capacity)
    def push(self, state, action_idx, reward, next_state, done): self.buffer.append((state, action_idx, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        return state, action, reward, next_state, done
    def __len__(self): return len(self.buffer)

def index_to_actions(index, num_drones):
    actions = []
    for _ in range(num_drones):
        actions.append(index % 6)
        index //= 6
    return actions

In [ ]:
def train_swarm_dqn(env, episodes=500, batch_size=128, gamma=0.99, lr=1e-3):
    obs_dim = env.observation_space.shape[0]
    num_drones = env.num_drones
    num_actions = 6 ** num_drones
    
    q_network = SwarmQNetwork(obs_dim, num_drones).to(device)
    target_network = SwarmQNetwork(obs_dim, num_drones).to(device)
    target_network.load_state_dict(q_network.state_dict())
    
    optimizer = optim.Adam(q_network.parameters(), lr=lr)
    buffer = ReplayBuffer(100000)
    
    epsilon = 1.0
    epsilon_decay = 0.99
    epsilon_min = 0.05
    
    rewards_history = []
    fire_counts_history = []
    
    print("Initiating Decentralized MARL Swarm Training (Teaching the AI Drones)...")
    
    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            if random.random() < epsilon:
                action_idx = random.randint(0, num_actions - 1)
            else:
                with torch.no_grad():
                    state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
                    q_values = q_network(state_tensor)
                    action_idx = q_values.argmax().item()
                    
            actions = index_to_actions(action_idx, num_drones)
            next_state, reward, done, _, info = env.step(actions)
            
            buffer.push(state, action_idx, reward, next_state, done)
            state = next_state
            total_reward += reward
            
            if len(buffer) > batch_size:
                s, a, r, s_next, d = buffer.sample(batch_size)
                s = torch.FloatTensor(s).to(device)
                a = torch.LongTensor(a).to(device)
                r = torch.FloatTensor(r).to(device)
                s_next = torch.FloatTensor(s_next).to(device)
                d = torch.FloatTensor(d).to(device)
                
                q_values = q_network(s)
                q_value = q_values.gather(1, a.unsqueeze(1)).squeeze(1)
                
                with torch.no_grad():
                    next_q_values = target_network(s_next)
                    max_next_q_values = next_q_values.max(1)[0]
                    target = r + gamma * max_next_q_values * (1 - d)
                    
                loss = nn.MSELoss()(q_value, target)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
        if episode % 10 == 0:
            target_network.load_state_dict(q_network.state_dict())
            
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards_history.append(total_reward)
        fire_counts_history.append(info['fire_count'])
        
        if (episode + 1) % 50 == 0:
            print(f"Episode {episode+1}/{episodes} | Total Reward (AI Score): {total_reward:.2f} | Remaining Fire: {info['fire_count']} cells | Epsilon (Randomness): {epsilon:.2f}")
            
    return q_network, rewards_history, fire_counts_history

In [ ]:
# Initialize Live Generative Environment: 10x10 Forest Grid, 3 Drones
env = WildfireSwarmEnv(grid_size=10, num_drones=3, max_payload=5, wind_vector=(1, 1))

# Train Model on Live Data
trained_model, rewards, fires = train_swarm_dqn(env, episodes=500)

# Plot Convergence Metrics (AI Learning Results)
fig, axs = plt.subplots(2, 1, figsize=(10, 8))

axs[0].plot(rewards, color='purple', alpha=0.7)
axs[0].set_title('MARL Swarm Convergence: Total Reward per Episode (AI Learning Score)')
axs[0].set_ylabel('Reward (Score)')
axs[0].grid(True, alpha=0.3)

axs[1].plot(fires, color='red', alpha=0.7)
axs[1].set_title('Suppression Efficacy: Remaining Fire Cells per Episode (Unextinguished Fire)')
axs[1].set_xlabel('Episode (Simulation Run)')
axs[1].set_ylabel('Active Fire Cells (Danger Level)')
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()